# Silver — Custos e Atendentes (`case.silver`)

Notebook de transformação da camada **Bronze → Silver** das tabelas `tb_custos` e `tb_base_atendentes`. O objetivo é entregar tabelas limpas, deduplicadas e padronizadas, prontas para consumo na camada Gold.

## Setup

Este bloco garante que o schema `case.silver` exista no Unity Catalog e define as variáveis de ambiente (`bronze_schema` e `silver_schema`) usadas em todo o notebook.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "case"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Schema {silver_schema} garantido. Iniciando transformações...")

In [0]:
df_bronze_custos = spark.table(f"{catalog}.bronze.tb_custos")
display(spark.table(f"{catalog}.bronze.tb_custos"))


## Transformação Bronze → Silver Custos

Este bloco lê a tabela `case.bronze.tb_custos` e aplica as regras de tratamento antes de gravar na camada Silver:

* **`tb_custos`**:
  * **Dedup**: remove duplicidades por `id_custo` (131.069 → 130.288 linhas), mantendo o registro mais recente via `ingestion_datetime`
  * **Tratamento de Valoração**: higieniza a coluna `custo` removendo textos e símbolos (`R$`, `reais`), substitui vírgulas (`,`) por pontos (`.`) e aplica a função `abs()` para inverter o sinal de valores negativos 
  * **Tipagem**: realiza o cast para `decimal(10,2)`


A gravação utiliza `overwrite` com `overwriteSchema`, garantindo idempotência e atualização dos schemas alterados.


In [0]:
df_bronze_custos = spark.table(f"{catalog}.bronze.tb_custos")
window_custos = Window.partitionBy("id_custo").orderBy(F.col("ingestion_datetime").desc())
df_silver_custos = ( df_bronze_custos
    .withColumn("id_custo", F.col("id_custo").cast("bigint"))
    .withColumn("id_chamado", F.col("id_chamado").cast("bigint"))
    .withColumn("custo", F.regexp_replace(F.col("custo"), "[^0-9,.-]", ""))
    .withColumn("custo", F.regexp_replace(F.col("custo"),",","."))
    .withColumn("custo",F.col("custo").cast("decimal(10,2)"))
    .withColumn("custo", F.abs(F.col("custo")))
    .withColumn("ranking", F.row_number().over(window_custos))
    .filter(F.col("ranking") == 1)
    .drop("ranking")
)


df_silver_custos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_custos")
print(f"Tabela: {silver_schema}.tb_custos atualizada com sucesso.")
print(f"  Bronze: {df_bronze_custos.count():,} linhas")
print(f"  Silver: {df_silver_custos.count():,} linhas")



## Debug — Validação de Qualidade e Integridade

Validação pós-transformação das tabelas `tb_custos`

* **`tb_custos`**: estatísticas descritivas (valores mínimo, máximo e médio), verificação de nulos e contagem de custos negativos 

* **Auditoria**: intervalo de datas da coluna `ingestion_datetime` 


In [0]:
df_custos = spark.table(f"{silver_schema}.tb_custos")

print("=" * 60)
print("DEBUG: tb_custos")
print("=" * 60)
print(f"Total de registros: {df_custos.count():,}")
print(f"Contagem de id_custo distintos: {df_custos.select('id_custo').distinct().count():,}")

print("\nEstatísticas da coluna 'custo':")
df_custos.select(
    F.min("custo").alias("custo_minimo"),
    F.max("custo").alias("custo_maximo"),
    F.round(F.avg("custo"), 2).alias("custo_medio"),
    F.count(F.when(F.col("custo").isNull(), 1)).alias("custos_nulos"),
    F.count(F.when(F.col("custo") < 0, 1)).alias("custos_negativos")
).show()
print(f"tb_custos ingestion_datetime min/max: {df_custos.agg(F.min('ingestion_datetime'), F.max('ingestion_datetime')).collect()[0]}")

## Transformação Bronze → Silver Atendentes

Este bloco lê a tabela `case.bronze.tb_base_atendentes` e aplica as regras de tratamento antes de gravar na camada Silver:

* **`tb_base_atendentes`**:
  * **`nivel_atendimento`**: padroniza a coluna `nivel_atendimento` para Nivel X sendo X = 1 ou X = 2
  * **Textos e Categóricas**: aplica `trim` em `nome_atendente` e `status_atendente`, e padroniza `status_atendente` em minúsculas (`ativo` / `inativo`)
  * **Datas**: preserva os tipos nativos `DateType` vindos da camada Bronze (`data_admissao` e `data_desligamento`)

In [0]:
df_bronze_atendentes = spark.table(f"{catalog}.bronze.tb_base_atendentes")
display(df_bronze_atendentes)

In [0]:
df_bronze_atendentes = spark.table(f"{catalog}.bronze.tb_base_atendentes")
window_atendentes = Window.partitionBy("id_atendente").orderBy(F.col("ingestion_datetime").desc())
df_silver_atendentes = (
    df_bronze_atendentes
    .withColumn("id_atendente", F.col("id_atendente").cast("bigint"))
    .withColumn("nome_atendente",F.trim(F.col("nome_atendente")))
    .withColumn("status_atendente",F.trim(F.lower(F.col("status_atendente"))))
    .withColumn("num_nivel", F.regexp_extract(F.col("nivel_atendimento"), r"(\d+)", 1))
    .withColumn("nivel_atendimento", F.concat(F.lit("N"), F.col("num_nivel")))
    .drop("num_nivel")
    .withColumn("ranking", F.row_number().over(window_atendentes))
    .filter(F.col("ranking") == 1)
    .drop("ranking")
)


df_silver_atendentes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_base_atendentes")

print(f"Tabelas no schema '{silver_schema}' atualizadas com sucesso!")

## Debug — Validação de Qualidade e Integridade
Validação pós-transformação das tabelas `tb_custos` e `tb_base_atendentes`:


* **`tb_base_atendentes`**: contagem de IDs únicos, distribuição de `nivel_atendimento` e validação do status dos atendentes
* **Auditoria**: intervalo de datas da coluna `ingestion_datetime` 

In [0]:
df_atendentes = spark.table(f"{silver_schema}.tb_base_atendentes")

print("\n" + "=" * 60)
print("DEBUG: tb_base_atendentes")
print("=" * 60)
print(f"Total de registros: {df_atendentes.count():,}")
print(f"Contagem de id_atendente distintos: {df_atendentes.select('id_atendente').distinct().count():,}")

print("\nValores distintos de 'nivel_atendimento':")
df_atendentes.groupBy("nivel_atendimento").count().orderBy("nivel_atendimento").show()

print("Valores distintos de 'status_atendente':")
df_atendentes.groupBy("status_atendente").count().show()

print("Contagem de Atendentes (Ativos vs Desligados):")
df_atendentes.select(
    F.count("id_atendente").alias("total_atendentes"),
    F.count(F.when(F.col("data_desligamento").isNull(), 1)).alias("ativos"),
    F.count(F.when(F.col("data_desligamento").isNotNull(), 1)).alias("desligados")
).show()

print("=" * 60)

print(f"tb_base_atendentes ingestion_datetime min/max: {df_atendentes.agg(F.min('ingestion_datetime'), F.max('ingestion_datetime')).collect()[0]}")
print("=" * 60)
